In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
from pandas import ExcelWriter
import datetime
from time import sleep
import os



In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'BD CBBAN' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.2.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

#writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running BD CBBAN Web Scraping Tool v.2.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



In [4]:
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate=now.strftime('%Y-%m-%d')

regdict={'BD CBBAN 1': 'Banks', 
'BD CBBAN 2': 'Finance Companies', 
'BD CBBAN 3': 'Micro Finance Institutions', 
'BD CBBAN 4': 'Others'}

Typology={'BD CBBAN 1': 'Banks', 
'BD CBBAN 2': 'Financial Institutions', 
'BD CBBAN 3': 'Micro Finance Institutions', 
'BD CBBAN 4': 'Others'}

In [5]:
def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

In [6]:

for reg in regdict:
	sleep(5)
	print(f'Working with {reg}.')
	driver.get('https://www.bb.org.bd/links/index.php')
	sleep(5)
	driver.find_element(By.CLASS_NAME, 'single-widget').find_element(By.LINK_TEXT, regdict[reg]).click()
	sleep(5)
	soup=BeautifulSoup(driver.page_source, 'html.parser')
	sleep(5)
	table=soup.find('table').find('tbody')
	for tr in table.find_all('tr')[1:]:#header in index 0
		#print(tr.text)
		sqldict['Name'].append(tr.find('td').text.strip())
		if tr.find('a', href=True):#in case of missing anchors/links
			sqldict['Website'].append(tr.find('a', href=True)['href'])
		sqldict['ListProcessDate'].append(processdate)
		sqldict['Cntry'].append('BD')
		sqldict['ListCode'].append(reg.split(' ')[-1])
		sqldict['RegulationType'].append('Regulated')
		sqldict['RegCtry'].append('BD')
		sqldict['RegCode'].append('CBBAN')
		sqldict['ListName'].append(Typology[reg])
		sqldict = bourange_same_length_array(sqldict)
		

    

Working with BD CBBAN 1.


AttributeError: 'NoneType' object has no attribute 'find'

In [ ]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 108 values.
Key 'priority' has 108 values.
Key 'ListLabel' has 108 values.
Key 'Typology' has 108 values.
Key 'EntryType' has 108 values.
Key 'Name' has 108 values.
Key 'InternalID_1' has 108 values.
Key 'InternalID_1_type' has 108 values.
Key 'InternalID_2' has 108 values.
Key 'InternalID_2_type' has 108 values.
Key 'InternalID_3' has 108 values.
Key 'InternalID_3_type' has 108 values.
Key 'CoType' has 108 values.
Key 'License_Type' has 108 values.
Key 'Address_1' has 108 values.
Key 'Address_2' has 108 values.
Key 'City' has 108 values.
Key 'Zip' has 108 values.
Key 'Cntry' has 108 values.
Key 'Phone' has 108 values.
Key 'Fax' has 108 values.
Key 'Website' has 108 values.
Key 'Email' has 108 values.
Key 'RegulationType' has 108 values.
Key 'RegulationTypeCode' has 108 values.
Key 'RegulationDate' has 108 values.
Key 'CancellationDate' has 108 values.
Key 'RegCtry' has 108 values.
Key 'RegCode' has 108 values.
Key 'ListCode' has 216 values.
Key 'ListLanguage' has 108 v

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)

sleep(3)

driver.quit()

ValueError: All arrays must be of the same length